# 02 · Auditoria da base de Despesa

**Esteira PASA:** `01 Receita` → `02 Despesa` → `03 Vidas` → `04 Consolidado` → Power BI / `relatorio_executivo.html`

## Objetivo do ficheiro bruto

`data/Despesa.csv` traz os **eventos assistenciais** (cerca de 5 milhões de linhas) com procedimento, classificação, regime de atendimento, `frequencia`, `valor_bruto` e `valor_coparticipacao`. É a base do custo assistencial: a **despesa líquida** usada no caderno 04 é `valor_bruto − valor_coparticipacao`.

Este caderno **não transforma nem grava** dados: valida tipos, integridade e domínios de valores e documenta a anomalia de coparticipação.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")


def localizar_raiz_projeto(inicio: Path | None = None) -> Path:
    """Sobe a arvore de diretorios ate encontrar a pasta ``data/`` do projeto."""
    inicio = (inicio or Path.cwd()).resolve()
    for pasta in (inicio, *inicio.parents):
        if (pasta / "data").is_dir():
            return pasta
    raise FileNotFoundError("Pasta 'data/' nao encontrada. Execute o caderno a partir da pasta do projeto.")


PASTA_RAIZ = localizar_raiz_projeto()
CHAVES_SEGMENTO = ["data_ref", "Operadora", "faixa_etaria", "uf", "cidade", "Plano"]

CAMINHO_DESPESA = PASTA_RAIZ / "data" / "Despesa.csv"
print(f"Fonte: {CAMINHO_DESPESA.relative_to(PASTA_RAIZ)}")

Fonte: data\Despesa.csv


In [2]:
def validar_schema(df: pd.DataFrame, esperado: dict[str, str]) -> pd.DataFrame:
    """Compara colunas e tipos lidos com o esperado; interrompe se houver divergencia."""
    relatorio = pd.DataFrame({
        "coluna": list(esperado),
        "tipo_esperado": list(esperado.values()),
        "tipo_lido": [str(df[c].dtype) if c in df else "AUSENTE" for c in esperado],
    })
    relatorio["confere"] = [
        c in df and (
            pd.api.types.is_string_dtype(df[c]) if tipo == "texto"
            else pd.api.types.is_integer_dtype(df[c]) if tipo == "inteiro"
            else pd.api.types.is_float_dtype(df[c]) if tipo == "decimal"
            else False
        )
        for c, tipo in esperado.items()
    ]
    assert list(df.columns) == list(esperado), f"Colunas divergentes: {list(df.columns)}"
    assert relatorio["confere"].all(), "Ha colunas com tipo diferente do esperado."
    return relatorio


def diagnosticar_integridade(df: pd.DataFrame, chave: list[str] | None) -> pd.Series:
    """Nulos, duplicidades, cobertura temporal e unicidade da chave de segmento."""
    datas = pd.to_datetime(df["data_ref"], format="%Y-%m-%d", errors="coerce")
    return pd.Series({
        "linhas": len(df),
        "celulas_nulas": int(df.isna().sum().sum()),
        "linhas_duplicadas_integralmente": int(df.duplicated().sum()),
        "segmentos_repetidos": int(df.duplicated(subset=chave).sum()) if chave else "n/a",
        "datas_invalidas": int(datas.isna().sum()),
        "primeira_competencia": datas.min().strftime("%m/%Y"),
        "ultima_competencia": datas.max().strftime("%m/%Y"),
        "competencias": datas.nunique(),
        "operadoras": ", ".join(sorted(df["Operadora"].unique())),
    }, name="valor")

## 1. Carga e validação de tipos

Dimensões como texto, `cod_procedimento` inteiro e as três medidas (`frequencia`, `valor_bruto`, `valor_coparticipacao`) como decimais.

In [3]:
df_despesa = pd.read_csv(CAMINHO_DESPESA)

SCHEMA_DESPESA = {
    "Indicador": "texto", "data_ref": "texto", "faixa_etaria": "texto", "uf": "texto",
    "cidade": "texto", "Operadora": "texto", "Plano": "texto", "cod_procedimento": "inteiro",
    "descricao_evento": "texto", "classificacao": "texto", "regime_atendimento": "texto",
    "frequencia": "decimal", "valor_bruto": "decimal", "valor_coparticipacao": "decimal",
}
display(validar_schema(df_despesa, SCHEMA_DESPESA))
display(df_despesa.head())

,coluna,tipo_esperado,tipo_lido,confere
0,Indicador,texto,str,True
1,data_ref,texto,str,True
2,faixa_etaria,texto,str,True
3,uf,texto,str,True
4,cidade,texto,str,True
5,Operadora,texto,str,True
6,Plano,texto,str,True
7,cod_procedimento,inteiro,int64,True
8,descricao_evento,texto,str,True
9,classificacao,texto,str,True


,Indicador,data_ref,faixa_etaria,uf,cidade,Operadora,Plano,cod_procedimento,descricao_evento,classificacao,regime_atendimento,frequencia,valor_bruto,valor_coparticipacao
0,Despesa,2026-06-01,59 e acima,RJ,RIO DE JANEIRO,Operadora Y,DBBECEB087,40202615,Endoscopia digestiva alta com biópsia e teste ...,EXAMES,Ambulatorial,1.00,275.00,0.00
1,Despesa,2025-08-01,0 - 18,MG,ITABIRA,Operadora X,71978B269B,50000616,Sessão individual ambulatorial de fonoaudiologia,TERAPIAS,Ambulatorial,665.00,"17,778.49","1,539.34"
2,Despesa,2025-08-01,39 - 43,PA,CANAA DOS CARAJAS,Operadora X,9889CA37E6,97020010,LENTE CORRETORA (PU/1A.P),MATERIAIS EM GERAL,Ambulatorial,4.00,629.25,756.91
3,Despesa,2026-04-01,0 - 18,MG,ITABIRA,Operadora X,71978B269B,50000080,"Sessão individual ambulatorial, em terapia ocu...",TERAPIAS,Ambulatorial,53.00,"3,327.20",615.18
4,Despesa,2025-08-01,0 - 18,GO,ITUMBIARA,Operadora X,BF1F47E31D,10101012,Consulta em consultório (no horário normal ou ...,CONSULTAS,Ambulatorial,1.00,100.00,150.00


## 2. Integridade estrutural

Em Despesa a repetição da chave de segmento é **esperada** (vários eventos por segmento), por isso o teste de unicidade de chave não se aplica.

In [4]:
integridade = diagnosticar_integridade(df_despesa, chave=None)
display(integridade.to_frame())

assert integridade["celulas_nulas"] == 0, "Ha valores nulos."
assert integridade["linhas_duplicadas_integralmente"] == 0, "Ha linhas duplicadas."
assert integridade["datas_invalidas"] == 0, "Ha datas invalidas."
assert integridade["competencias"] == 12, "Esperadas 12 competencias (08/2025 a 07/2026)."
assert df_despesa["Indicador"].eq("Despesa").all()

,valor
linhas,4995231
celulas_nulas,0
linhas_duplicadas_integralmente,0
segmentos_repetidos,n/a
datas_invalidas,0
primeira_competencia,08/2025
ultima_competencia,07/2026
competencias,12
operadoras,"Operadora X, Operadora Y"


## 3. Domínio dos valores

Verificamos negativos, zeros e frequências fracionárias. A frequência é usada no caderno 04 como *proxy* de utilização (itens por vida), por isso precisa ser essencialmente inteira.

In [5]:
colunas_valor = ["frequencia", "valor_bruto", "valor_coparticipacao"]
display(df_despesa[colunas_valor].describe().T)

dominio = pd.DataFrame({
    "negativos": (df_despesa[colunas_valor] < 0).sum(),
    "zerados": (df_despesa[colunas_valor] == 0).sum(),
})
dominio["zerados_pct"] = dominio["zerados"] / len(df_despesa) * 100
display(dominio)

frequencia_fracionaria_pct = (df_despesa["frequencia"] % 1 != 0).mean() * 100
print(f"Eventos com frequencia fracionaria: {frequencia_fracionaria_pct:.2f}%")

assert dominio["negativos"].eq(0).all(), "Ha valores negativos nas medidas."
assert frequencia_fracionaria_pct < 0.1, "Frequencia fracionaria acima do tolerado para uso como proxy."

,count,mean,std,min,25%,50%,75%,max
frequencia,"4,995,231.00",6.37,124.32,0.00,1.00,1.00,3.00,"74,000.00"
valor_bruto,"4,995,231.00",196.96,"1,864.22",0.00,6.53,20.63,75.00,"986,540.76"
valor_coparticipacao,"4,995,231.00",26.44,263.36,0.00,0.55,3.03,11.27,"203,523.16"


,negativos,zerados,zerados_pct
frequencia,0,0,0.00
valor_bruto,0,22,0.00
valor_coparticipacao,0,522669,10.46


Eventos com frequencia fracionaria: 0.05%


In [6]:
resumo_operadora = df_despesa.groupby("Operadora").agg(
    eventos=("valor_bruto", "size"),
    planos=("Plano", "nunique"),
    despesa_bruta=("valor_bruto", "sum"),
    coparticipacao=("valor_coparticipacao", "sum"),
)
resumo_operadora["coparticipacao_pct_bruta"] = (
    resumo_operadora["coparticipacao"] / resumo_operadora["despesa_bruta"] * 100
)
display(resumo_operadora)
display(df_despesa["classificacao"].value_counts().to_frame("eventos"))

,eventos,planos,despesa_bruta,coparticipacao,coparticipacao_pct_bruta
Operadora,,,,,
Operadora X,3985249,162,"689,565,995.16","96,409,469.03",13.98
Operadora Y,1009982,16,"294,291,646.12","35,665,868.56",12.12


,eventos
classificacao,
EXAMES,2785977
MEDICAMENTOS,705152
MATERIAIS,674905
DIÁRIAS/ TAXAS,215861
HONORÁRIOS MÉDICOS,214786
CONSULTAS,159120
PACOTE,97749
TERAPIAS,67069
MATERIAIS EM GERAL,24717


## 4. Achado crítico — coparticipação superior ao valor bruto

Por definição, a coparticipação é uma **parcela** do valor do evento paga pelo beneficiário. Registros em que `valor_coparticipacao > valor_bruto` geram **despesa líquida negativa**, o que não tem interpretação assistencial direta.

In [7]:
anomalia = df_despesa["valor_coparticipacao"] > df_despesa["valor_bruto"]
saldo = (df_despesa["valor_bruto"] - df_despesa["valor_coparticipacao"]).where(anomalia)

anomalia_operadora = pd.DataFrame({
    "eventos": df_despesa.groupby("Operadora").size(),
    "ocorrencias": anomalia.groupby(df_despesa["Operadora"]).sum(),
    "saldo_negativo": saldo.groupby(df_despesa["Operadora"]).sum(),
})
anomalia_operadora["ocorrencias_pct"] = anomalia_operadora["ocorrencias"] / anomalia_operadora["eventos"] * 100
display(anomalia_operadora)

print(f"Total de ocorrencias: {int(anomalia.sum()):,}".replace(",", "."))
display(df_despesa.loc[anomalia, "classificacao"].value_counts().head(6).to_frame("ocorrencias"))

assert int(anomalia.sum()) == 22_345, "Contagem de ocorrencias mudou: revisar a narrativa."
assert anomalia_operadora.at["Operadora X", "ocorrencias_pct"] > anomalia_operadora.at["Operadora Y", "ocorrencias_pct"]

,eventos,ocorrencias,saldo_negativo,ocorrencias_pct
Operadora,,,,
Operadora X,3985249,21188,"-5,056,366.62",0.53
Operadora Y,1009982,1157,"-639,702.86",0.11


Total de ocorrencias: 22.345


,ocorrencias
classificacao,
CONSULTAS,5599
EXAMES,4760
TERAPIAS,3893
MATERIAIS EM GERAL,3751
HONORÁRIOS MÉDICOS,2432
DIÁRIAS/ TAXAS,1164


**Evidência:** 22.345 eventos têm coparticipação acima do valor bruto — **0,53% dos eventos da X** e **0,11% da Y** (taxa cerca de 5× maior na X), concentrados em consultas, exames, terapias e materiais. O saldo negativo soma cerca de R$ 5,7 milhões.

**Risco:** a taxa desigual entre operadoras sugere uma **regra de lançamento não documentada** (estornos, coparticipação agregada por guia, teto de coparticipação) e não erro aleatório. Sem dicionário de dados, a definição de despesa líquida fica ambígua.

**Recomendação / tratamento adotado no caderno 04:** os registros são **preservados e sinalizados** (nunca excluídos nem truncados). O caderno 04 mede a sensibilidade: truncar os negativos em zero move a sinistralidade da Y de 34,15% para 34,24% — efeito **imaterial**. Recomenda-se auditar a regra de coparticipação com a operadora e publicar um dicionário de dados.

## 5. Conclusões da auditoria

| Verificação | Resultado |
|---|---|
| Colunas e tipos | Conforme o esperado (14 colunas) |
| Nulos, duplicados, datas inválidas | Nenhum |
| Valores negativos | Nenhum nas medidas originais |
| Frequência fracionária | 0,05% dos eventos — aceitável como *proxy* de utilização |
| Coparticipação > bruto | **22.345 eventos** (X 0,53%; Y 0,11%) — preservados e testados por sensibilidade |
| Cobertura | Operadoras X e Y, 12 competências (08/2025 a 07/2026) |